# StreamLoop churn model tuning

StreamLoop wants to spot customers who are about to cancel. Missing a customer who will leave is more costly than sending a stay offer to someone who would have stayed.

This notebook tunes one logistic regression model. It starts with factory settings, tries a wide spread of settings, then looks more closely at the settings that did well. The locked-away test customers are scored twice: once for the factory model, and once for the final model. The searches see only the practice customers.

The customer table is a public phone-company list used as a stand-in for StreamLoop. It is loaded from the project URL.


## The score we care about

Recall for quitters answers a direct question: of the customers who really left, how many did we catch?

Accuracy can look strong when a model mostly says "this person will stay," because most customers stay. Accuracy is written down as extra context. Precision, F1, the confusion-matrix counts, and PR-AUC are written down too. They show the burden of the extra stay offers. They do not choose the winner.

`Churn == "Yes"` is the positive class. In this notebook that label becomes `1`.


## Setup

The random seed is 42 for the split, the classifier, and the broad search. The practice pile is 80 percent of the rows. The test pile is 20 percent. Both piles keep the same share of quitters. Each search checks its settings with 5 shuffled folds that also keep that share.

Every logistic regression uses `max_iter=1000` and `random_state=42`, including the factory model. Those two controls make the fit finish and make the run repeatable. The factory model still uses the factory regularization strength `C=1` and no class weight.

This computer's Python does not trust the download website's certificate on its own. The notebook points HTTPS at the `certifi` certificate list, then still reads the table from the project URL. No local copy of the table is required.


In [1]:
import json
import os
import sys
from pathlib import Path

import certifi
import pandas as pd
import sklearn
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocess import TotalChargesCleaner

os.environ["SSL_CERT_FILE"] = certifi.where()

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_FOLDS = 5
DATA_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("certifi", certifi.__version__)
print("scipy loguniform is available")
print("seed", RANDOM_STATE)


pandas 3.1.1
scikit-learn 1.9.1
certifi 2026.10.23
scipy loguniform is available
seed 42


## Load the customer list and check the contract

The table should have 7,043 customers and 21 columns. `customerID` is an ID tag, so it will not be a model feature. The target should contain only `Yes` and `No`.

Eleven `TotalCharges` cells are spaces rather than numbers. Those same rows have `tenure == 0`. The raw file does not mark them as missing yet. Phrases such as `No internet service` and `No phone service` are real answers, so they stay as categories.


In [2]:
EXPECTED_COLUMNS = [
    "customerID",
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
]
EXPECTED_CATEGORIES = {
    "gender": ["Female", "Male"],
    "SeniorCitizen": ["0", "1"],
    "Partner": ["No", "Yes"],
    "Dependents": ["No", "Yes"],
    "PhoneService": ["No", "Yes"],
    "MultipleLines": ["No", "No phone service", "Yes"],
    "InternetService": ["DSL", "Fiber optic", "No"],
    "OnlineSecurity": ["No", "No internet service", "Yes"],
    "OnlineBackup": ["No", "No internet service", "Yes"],
    "DeviceProtection": ["No", "No internet service", "Yes"],
    "TechSupport": ["No", "No internet service", "Yes"],
    "StreamingTV": ["No", "No internet service", "Yes"],
    "StreamingMovies": ["No", "No internet service", "Yes"],
    "Contract": ["Month-to-month", "One year", "Two year"],
    "PaperlessBilling": ["No", "Yes"],
    "PaymentMethod": [
        "Bank transfer (automatic)",
        "Credit card (automatic)",
        "Electronic check",
        "Mailed check",
    ],
    "Churn": ["No", "Yes"],
}

customers = pd.read_csv(DATA_URL)

if list(customers.columns) != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected columns: {list(customers.columns)}")
if set(customers["Churn"].unique()) != {"Yes", "No"}:
    raise ValueError("Churn must contain only Yes and No.")
if not customers["customerID"].is_unique:
    raise ValueError("customerID values are not unique.")
if len(customers) != 7043:
    raise ValueError(f"Expected 7,043 rows, found {len(customers)}.")

blank_charges = customers["TotalCharges"].astype(str).str.strip().eq("")
if int(blank_charges.sum()) != 11:
    raise ValueError("Expected 11 blank TotalCharges values.")
if int((blank_charges & customers["tenure"].eq(0)).sum()) != 11:
    raise ValueError("Blank TotalCharges rows should also have tenure 0.")

unexpected_categories = {}
for column, expected in EXPECTED_CATEGORIES.items():
    found = sorted(customers[column].astype(str).unique())
    if found != expected:
        unexpected_categories[column] = found
if unexpected_categories:
    raise ValueError(f"Unexpected categories: {unexpected_categories}")

class_counts = customers["Churn"].value_counts()
print("rows", len(customers), "columns", customers.shape[1])
print("class counts before modeling")
print(class_counts.to_string())
print("duplicate rows", int(customers.duplicated().sum()))
print("raw missing values", int(customers.isna().sum().sum()))
print("blank TotalCharges", int(blank_charges.sum()))
print("unexpected categories: none")
print("customerID is unique and will be left out of the features")


rows 7043 columns 21
class counts before modeling
Churn
No     5174
Yes    1869
duplicate rows 0
raw missing values 0
blank TotalCharges 11
unexpected categories: none
customerID is unique and will be left out of the features


## Lock away the test customers

The split happens before any filler, scaler, or encoder is trained. Blank charges are still text at this point. Turning them into missing numbers happens later, inside the pipeline, so that step is copied correctly during cross-validation.


In [3]:
target = (customers["Churn"] == "Yes").astype(int)
features = customers.drop(columns=["Churn", "customerID"])

if "customerID" in features.columns:
    raise ValueError("customerID is still in the features.")

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=TEST_SIZE,
    stratify=target,
    random_state=RANDOM_STATE,
)

print("practice rows", len(X_train), "test rows", len(X_test))
print("practice quitter share", round(float(y_train.mean()), 4))
print("test quitter share", round(float(y_test.mean()), 4))
print("practice features", list(X_train.columns))


practice rows 5634 test rows 1409
practice quitter share 0.2654
test quitter share 0.2654
practice features ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


## One pipeline

The pipeline does four jobs, in order:

1. Turn blank `TotalCharges` into missing numbers. This step does not learn a fill value.
2. Fill missing numbers with the practice-set median, then scale those numbers. Logistic regression treats a larger number as a stronger signal, so tenure and the two charge columns need a shared scale.
3. Fill missing categories with the most common practice-set value, then turn each category into yes/no columns. A category that shows up later is ignored instead of crashing the run.
4. Fit logistic regression.

`SeniorCitizen` is a yes/no flag stored as 0 or 1, so it is handled with the categories.

The same pipeline definition is used for the factory model and both searches. The factory model is a copy, so fitting it does not change the copy used for the search.


In [4]:
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]


def make_pipeline():
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, NUMERIC_FEATURES),
            ("categorical", categorical_pipe, CATEGORICAL_FEATURES),
        ]
    )
    return Pipeline(
        steps=[
            ("cleaner", TotalChargesCleaner()),
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            ),
        ]
    )


pipeline = make_pipeline()
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
print(pipeline)


Pipeline(steps=[('cleaner', TotalChargesCleaner()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                               

## Factory-settings model

The factory model uses `C=1` and no class weight. It is trained on the practice customers only. The locked test customers are scored once in this section.

Recall is the main number. The other numbers use the same predictions, so they are part of this one scoring event rather than extra peeks used to change the model.


In [5]:
TEST_SET_USES = []


def score_locked_test(stage_name, model):
    if stage_name not in {"baseline", "final"}:
        raise ValueError("The test set has only two allowed scoring stages.")
    if stage_name in TEST_SET_USES or len(TEST_SET_USES) >= 2:
        raise RuntimeError("The test set can be scored only twice.")
    classes = list(model.named_steps["classifier"].classes_)
    if classes != [0, 1]:
        raise ValueError(f"Expected classes [0, 1], found {classes}.")

    predictions = model.predict(X_test)
    quit_probability = model.predict_proba(X_test)[:, 1]
    confusion = confusion_matrix(y_test, predictions, labels=[0, 1])
    true_negative, false_positive, false_negative, true_positive = confusion.ravel()
    TEST_SET_USES.append(stage_name)
    return {
        "recall": float(recall_score(y_test, predictions, pos_label=1)),
        "precision": float(precision_score(y_test, predictions, pos_label=1, zero_division=0)),
        "f1": float(f1_score(y_test, predictions, pos_label=1, zero_division=0)),
        "pr_auc": float(average_precision_score(y_test, quit_probability)),
        "accuracy": float(accuracy_score(y_test, predictions)),
        "true_negative": int(true_negative),
        "false_positive": int(false_positive),
        "false_negative": int(false_negative),
        "true_positive": int(true_positive),
    }


def show_scores(title, scores):
    print(title)
    print(f"recall (main score): {scores['recall']:.4f}")
    print(f"precision: {scores['precision']:.4f}")
    print(f"f1: {scores['f1']:.4f}")
    print(f"pr_auc: {scores['pr_auc']:.4f}")
    print(f"accuracy (context only): {scores['accuracy']:.4f}")
    print(
        "confusion matrix counts for quitters as the positive class: "
        f"true negatives {scores['true_negative']}, "
        f"false positives {scores['false_positive']}, "
        f"false negatives {scores['false_negative']}, "
        f"true positives {scores['true_positive']}"
    )


baseline = clone(pipeline)
baseline.fit(X_train, y_train)
baseline_scores = score_locked_test("baseline", baseline)
show_scores("Factory model on the locked test customers", baseline_scores)


Factory model on the locked test customers
recall (main score): 0.5588
precision: 0.6572
f1: 0.6040
pr_auc: 0.6338
accuracy (context only): 0.8055
confusion matrix counts for quitters as the positive class: true negatives 926, false positives 109, false negatives 165, true positives 209


## A category the model has not seen

This check builds one fake practice row with a contract name that was not in the training rows. It does not look at the test pile. The encoder is set to ignore unknown categories, so this row should transform instead of raising an error.


In [6]:
unseen_row = X_train.iloc[[0]].copy()
unseen_row["Contract"] = "brand-new-contract"
unseen_cleaned = baseline.named_steps["cleaner"].transform(unseen_row)
unseen_ready = baseline.named_steps["preprocessor"].transform(unseen_cleaned)
if unseen_ready.shape[0] != 1:
    raise ValueError("The unseen-category row did not transform.")
print("unseen category transformed with shape", tuple(unseen_ready.shape))


unseen category transformed with shape (1, 46)


## Broad search

The first search tries 20 random pairs. `C` is drawn between 0.001 and 100. Class weight is either off or `balanced`. Balanced tells the model to pay more attention to quitters, who are the smaller group.

The search scores recall. It uses the 5 practice folds, all processor cores, seed 42, and `refit=True`. With `refit=True`, scikit-learn retrains the highest-average settings on the whole practice pile. This notebook does not call `fit` on that retrained estimator. The broad-search winner is not the final model. It only shows where the tight search should look.

`C` and class weight are always a legal pair for this classifier, so the search does not have to skip broken combinations.


In [7]:
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions={
        "classifier__C": loguniform(1e-3, 1e2),
        "classifier__class_weight": [None, "balanced"],
    },
    n_iter=20,
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    return_train_score=True,
)
random_search.fit(X_train, y_train)
print("broad search fitted on practice rows only:", len(X_train))
print("best average settings from the broad search:", random_search.best_params_)
print("best average recall:", round(float(random_search.best_score_), 4))


broad search fitted on practice rows only: 5634
best average settings from the broad search: {'classifier__C': np.float64(3.4702669886504163), 'classifier__class_weight': 'balanced'}
best average recall: 0.802


In [8]:
def results_frame(search):
    raw = search.cv_results_
    records = []
    for index, params in enumerate(raw["params"]):
        weight = params["classifier__class_weight"]
        record = {
            "rank": int(raw["rank_test_score"][index]),
            "C": float(params["classifier__C"]),
            "class_weight": "None" if weight is None else str(weight),
            "mean_recall": float(raw["mean_test_score"][index]),
            "std_recall": float(raw["std_test_score"][index]),
            "mean_train_recall": float(raw["mean_train_score"][index]),
            "mean_fit_seconds": float(raw["mean_fit_time"][index]),
        }
        for fold in range(N_FOLDS):
            record[f"fold_{fold + 1}_recall"] = float(raw[f"split{fold}_test_score"][index])
        records.append(record)
    frame = pd.DataFrame.from_records(records)
    return frame.sort_values(["rank", "std_recall"]).reset_index(drop=True)


random_results = results_frame(random_search)
summary_columns = [
    "rank",
    "C",
    "class_weight",
    "mean_recall",
    "std_recall",
    "mean_train_recall",
    "mean_fit_seconds",
]
print("All 20 broad-search settings, best average recall first")
print(random_results[summary_columns].round(4).to_string(index=False))
print()
print("Fold-by-fold recall for the five leading settings")
fold_columns = [f"fold_{fold}_recall" for fold in range(1, N_FOLDS + 1)]
print(random_results.head(5)[summary_columns[:5] + fold_columns].round(4).to_string(index=False))


All 20 broad-search settings, best average recall first
 rank       C class_weight  mean_recall  std_recall  mean_train_recall  mean_fit_seconds
    1  3.4703     balanced       0.8020      0.0376             0.8060            0.3581
    2  0.3725     balanced       0.8013      0.0370             0.8050            0.3687
    3  1.2357     balanced       0.8007      0.0370             0.8069            0.3721
    4 14.5282     balanced       0.8000      0.0379             0.8052            0.3696
    5  0.0466     balanced       0.7953      0.0326             0.8012            0.3556
    6  0.0010     balanced       0.7926      0.0302             0.7952            0.2782
    7  0.0020     balanced       0.7920      0.0332             0.7930            0.2873
    7  0.0019     balanced       0.7920      0.0332             0.7933            0.2718
    9  0.0083     balanced       0.7900      0.0316             0.7945            0.3004
    9  0.0083     balanced       0.7900      0.0316   

## Why the tight grid looks like this

The broad search split into two clear groups.

- With class weight `balanced`, practice recall sat around 0.79 to 0.80. The best draw was about `C=3.47`, with mean recall 0.802 and a fold spread of about 0.038. Draws near `C=0.37`, `C=1.24`, and `C=14.5` were within about 0.002 of that best average. That gap is tiny next to the fold spread.
- With class weight off, practice recall sat around 0.54 to 0.55. That is about 0.25 below the balanced group, several times the fold spread. Class weight off is the simpler choice, and it is not a tie. It misses far more quitters.

The tight grid therefore stays on `balanced`. It checks the flat ridge of `C` with round values that cover the strong draws, including the factory value 1:

`0.3, 0.5, 1, 2, 3.5, 5, 10, 15`

The next code cell checks that this reading of the broad search is still true before the tight grid runs.


## How the winner is chosen

`refit=True` always keeps the single highest average. On the tight grid, a tiny lead inside the fold-to-fold noise is not treated as a real win. The tight search therefore uses a selection function. Scikit-learn still trains the chosen settings once on the full practice pile. This notebook does not call `fit` on that chosen estimator afterward.

A setting is tied with the leader when its average recall is no worse than the leader by more than the leader's fold-to-fold spread. Among tied settings, class weight off wins if it is actually in the tie. Otherwise the tied setting with `C` closest to the factory value 1 wins. If that still ties, the smaller fold spread wins.


In [9]:
balanced_band = random_results[
    (random_results["class_weight"] == "balanced")
    & random_results["C"].between(0.3, 15)
]
unbalanced = random_results[random_results["class_weight"] == "None"]
if balanced_band.empty or unbalanced.empty:
    raise ValueError("The broad search did not produce both groups needed to build the grid.")
if balanced_band["mean_recall"].min() <= unbalanced["mean_recall"].max():
    raise ValueError("Class weight off was not clearly worse. The tight grid needs another look.")
band_gap = balanced_band["mean_recall"].max() - balanced_band["mean_recall"].min()
if band_gap >= 0.02:
    raise ValueError("C from 0.3 to 15 was not a flat ridge. The tight grid needs another look.")
print(
    "Broad-search check passed. Balanced C values from 0.3 to 15 stay within "
    f"{band_gap:.4f} recall, and every unbalanced setting is lower."
)

FOCUSED_C_VALUES = [0.3, 0.5, 1.0, 2.0, 3.5, 5.0, 10.0, 15.0]


def choose_index(cv_results):
    """Pick a tied simpler setting, otherwise the highest average."""
    means = [float(value) for value in cv_results["mean_test_score"]]
    spreads = [float(value) for value in cv_results["std_test_score"]]
    leader = max(range(len(means)), key=lambda index: means[index])
    cutoff = means[leader] - spreads[leader]
    tied = [index for index, mean in enumerate(means) if mean >= cutoff]

    def sort_key(index):
        params = cv_results["params"][index]
        weight = params["classifier__class_weight"]
        weight_rank = 0 if weight is None else 1
        distance_from_factory_c = abs(float(params["classifier__C"]) - 1.0)
        return (weight_rank, distance_from_factory_c, spreads[index], -means[index])

    return int(min(tied, key=sort_key))


grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid={
        "classifier__C": FOCUSED_C_VALUES,
        "classifier__class_weight": ["balanced"],
    },
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    refit=choose_index,
    return_train_score=True,
)
grid_search.fit(X_train, y_train)
if not hasattr(grid_search.best_estimator_.named_steps["classifier"], "coef_"):
    raise RuntimeError("The search did not train the chosen estimator.")
print("tight search fitted on practice rows only:", len(X_train))
print("chosen settings:", grid_search.best_params_)


Broad-search check passed. Balanced C values from 0.3 to 15 stay within 0.0020 recall, and every unbalanced setting is lower.


tight search fitted on practice rows only: 5634
chosen settings: {'classifier__C': 1.0, 'classifier__class_weight': 'balanced'}


In [10]:
grid_results = results_frame(grid_search)
print("Tight-grid settings, with every practice fold")
print(
    grid_results[summary_columns + fold_columns]
    .round(4)
    .to_string(index=False)
)

leader_row = grid_results.sort_values(["mean_recall", "std_recall"], ascending=[False, True]).iloc[0]
chosen_c = float(grid_search.best_params_["classifier__C"])
chosen_weight = grid_search.best_params_["classifier__class_weight"]
chosen_label = "None" if chosen_weight is None else str(chosen_weight)
chosen_row = grid_results[
    (grid_results["C"] == chosen_c) & (grid_results["class_weight"] == chosen_label)
].iloc[0]

leader_mean = float(leader_row["mean_recall"])
leader_std = float(leader_row["std_recall"])
chosen_mean = float(chosen_row["mean_recall"])
chosen_std = float(chosen_row["std_recall"])
mean_gap = leader_mean - chosen_mean
chosen_is_leader = bool(
    chosen_c == float(leader_row["C"]) and chosen_label == leader_row["class_weight"]
)

print()
if chosen_is_leader:
    decision_text = (
        f"The highest average recall is {leader_mean:.4f} "
        f"at C={float(leader_row['C'])} with class weight {leader_row['class_weight']}. "
        f"Its fold spread is {leader_std:.4f}. "
        "No simpler tied setting beat that choice under the rule, so this is the final model."
    )
else:
    decision_text = (
        f"The highest average recall is {leader_mean:.4f} "
        f"at C={float(leader_row['C'])} with class weight {leader_row['class_weight']}. "
        f"Its fold spread is {leader_std:.4f}. "
        f"The chosen settings, C={chosen_c} and class weight {chosen_label}, "
        f"have average recall {chosen_mean:.4f} and fold spread {chosen_std:.4f}. "
        f"The gap is {mean_gap:.4f}, which sits inside the leader's fold spread, so it is not a real win. "
        "Class weight stays balanced because turning it off was not in the tie. "
        f"C={chosen_c} is the tied setting closest to the factory value 1."
    )
print(decision_text)


Tight-grid settings, with every practice fold
 rank    C class_weight  mean_recall  std_recall  mean_train_recall  mean_fit_seconds  fold_1_recall  fold_2_recall  fold_3_recall  fold_4_recall  fold_5_recall
    1  3.5     balanced       0.8020      0.0376             0.8062            0.3824         0.8261         0.7458         0.7993         0.8562         0.7826
    2  2.0     balanced       0.8013      0.0366             0.8067            0.4186         0.8261         0.7458         0.7993         0.8528         0.7826
    3  0.3     balanced       0.8013      0.0370             0.8050            0.3585         0.8261         0.7458         0.8027         0.8528         0.7793
    3  1.0     balanced       0.8013      0.0379             0.8062            0.3766         0.8261         0.7458         0.7993         0.8562         0.7793
    5  0.5     balanced       0.8007      0.0370             0.8052            0.3736         0.8261         0.7458         0.7993         0.8528    

## Final score on the locked test customers

This is the second and last time the test pile is used. The metrics match the factory-model scorecard. The chosen estimator was already trained by the tight search, so it is not trained again here.


In [11]:
final_model = grid_search.best_estimator_
final_scores = score_locked_test("final", final_model)
show_scores("Tuned model on the locked test customers", final_scores)

comparison = pd.DataFrame(
    [baseline_scores, final_scores],
    index=["factory", "tuned"],
)
print()
print("Same scorecard for both models")
print(comparison.round(4).to_string())
print()
print("test set scoring stages:", TEST_SET_USES)
if TEST_SET_USES != ["baseline", "final"]:
    raise RuntimeError("The test set was not scored in the required two stages.")

summary = {
    "practice_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "quitters": int(class_counts["Yes"]),
    "stayers": int(class_counts["No"]),
    "blank_total_charges": 11,
    "baseline": {key: round(value, 4) if isinstance(value, float) else value for key, value in baseline_scores.items()},
    "tuned": {key: round(value, 4) if isinstance(value, float) else value for key, value in final_scores.items()},
    "chosen_C": chosen_c,
    "chosen_class_weight": chosen_label,
    "chosen_mean_recall": round(chosen_mean, 4),
    "chosen_std_recall": round(chosen_std, 4),
    "leader_C": float(leader_row["C"]),
    "leader_class_weight": str(leader_row["class_weight"]),
    "leader_mean_recall": round(leader_mean, 4),
    "leader_std_recall": round(leader_std, 4),
    "mean_gap": round(mean_gap, 4),
    "chosen_is_highest_mean": chosen_is_leader,
    "decision_text": decision_text,
}
print("METRIC_SUMMARY_START")
print(json.dumps(summary, indent=2))
print("METRIC_SUMMARY_END")


Tuned model on the locked test customers
recall (main score): 0.7834
precision: 0.5034
f1: 0.6130
pr_auc: 0.6324
accuracy (context only): 0.7374
confusion matrix counts for quitters as the positive class: true negatives 746, false positives 289, false negatives 81, true positives 293

Same scorecard for both models
         recall  precision     f1  pr_auc  accuracy  true_negative  false_positive  false_negative  true_positive
factory  0.5588     0.6572  0.604  0.6338    0.8055            926             109             165            209
tuned    0.7834     0.5034  0.613  0.6324    0.7374            746             289              81            293

test set scoring stages: ['baseline', 'final']
METRIC_SUMMARY_START
{
  "practice_rows": 5634,
  "test_rows": 1409,
  "quitters": 1869,
  "stayers": 5174,
  "blank_total_charges": 11,
  "baseline": {
    "recall": 0.5588,
    "precision": 0.6572,
    "f1": 0.604,
    "pr_auc": 0.6338,
    "accuracy": 0.8055,
    "true_negative": 926,
    